# Predictive Maintenance 

This assignment covers the topic of predictive maintenance. Predictive Maintenance problems adress predicting when a machine needs to be maintained ahead of breaking down. This problem can occur anywhere regular maintenance is required for a machine. For example, it can be used in manufacturing, fleet operations, train maintenance, etc.

This assignment will use the [Predictive Maintenance Dataset](https://archive.ics.uci.edu/ml/datasets/AI4I+2020+Predictive+Maintenance+Dataset). The dataset consists of 10 000 data points stored as rows with 14 features in columns. The 'machine failure' label that indicates, whether the machine has failed in this particular datapoint.

# Learning Objectives
- Perform model tuning based on hyper parameters.
- Select the best model after attempting multiple models.
- Perform recursive feature elimination, producing a statistically significant improvement over a model without feature selection.

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing, metrics
from sklearn.model_selection import train_test_split


ai4i2020 = pd.read_csv('ai4i2020.csv')
print(ai4i2020.info())
ai4i2020.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  object 
 4   Process temperature [K]  10000 non-null  object 
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 703.2+ KB
None


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,1,M14860,M,298.1,308.6,1551,42.8,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0
5,6,M14865,M,298.1,308.6,1425,41.9,11,0
6,7,L47186,L,298.1,308.6,1558,42.4,14,0
7,8,L47187,L,298.1,308.6,1527,40.2,16,0
8,9,M14868,M,298.3,308.7,1667,28.6,18,0
9,10,M14869,M,298.5,309,1741,28.0,21,0


Question 1.1:  Write a command that will calculate the number of unique values for each feature in the training data.

In [2]:
df = ai4i2020

In [3]:
# Command(s)
for feature in df.columns.tolist(): print("The number of unique values for {} is {}.\n".format(feature, df[feature].nunique()))

The number of unique values for UDI is 10000.

The number of unique values for Product ID is 10000.

The number of unique values for Type is 3.

The number of unique values for Air temperature [K] is 93.

The number of unique values for Process temperature [K] is 82.

The number of unique values for Rotational speed [rpm] is 941.

The number of unique values for Torque [Nm] is 577.

The number of unique values for Tool wear [min] is 246.

The number of unique values for Machine failure is 2.



Question 1.2: Determine if the data contains any missing values, and replace the values with np.nan. Missing values would be '?'.

In [4]:
for feature in df.columns.tolist()[1:]: print(df[feature].value_counts().transpose(),"\n Number of Elements Total:",len(df[feature]),"\t Number of Unique Elements:",df[feature].value_counts().sum())

M14860    1
L53850    1
L53843    1
L53844    1
L53845    1
         ..
M18193    1
M18194    1
L50515    1
L50516    1
M24859    1
Name: Product ID, Length: 10000, dtype: int64 
 Number of Elements Total: 10000 	 Number of Unique Elements: 10000
L    6000
M    2997
H    1003
Name: Type, dtype: int64 
 Number of Elements Total: 10000 	 Number of Unique Elements: 10000
300.7    279
298.9    231
297.4    230
300.5    229
298.8    227
        ... 
304.4      7
296        6
295.4      3
295.3      3
304.5      1
Name: Air temperature [K], Length: 93, dtype: int64 
 Number of Elements Total: 10000 	 Number of Unique Elements: 10000
310.6    317
310.8    273
310.7    266
308.6    265
310.5    263
        ... 
306.9      4
313.7      4
305.8      3
305.7      2
313.8      2
Name: Process temperature [K], Length: 82, dtype: int64 
 Number of Elements Total: 10000 	 Number of Unique Elements: 10000
1452    48
1435    43
1447    42
1429    40
1469    40
        ..
2197     1
2211     1
1905     

In [5]:
#There doesn't seem to be any missing values but
df = df.replace(regex="\?$",value=np.NAN) #would replace any with NaN if they existed.

Question 1.3: Replace all missing values with the mean. Change column types to numeric.

In [7]:
#Again there doesn't appear to be any missing values but here we set all instances of NaN to the feature's average.
for feature in df.columns.tolist()[3:]: df[feature]=df[feature].astype(float)
for feature in df.columns.tolist()[3:-1]: df[feature]=df[feature].fillna(value=df[feature].mean())


Question 1.4: Drop UDI and 'Product ID' from the data

In [8]:
df = df.drop(columns=["UDI","Product ID"])

Question 2.1: Split the data into training and testing taking into consideration 'Machine failure' as the target (y)

In [9]:
X, y = df.drop(columns="Machine failure"), df["Machine failure"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
bak = [X_train, X_test, y_train, y_test]

Question 2.2: Apply [One-Hot Encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) to data. Make sure to Fit the training data and transform both training and test data. 

In [12]:
from sklearn.preprocessing import OneHotEncoder

In [13]:
enc = OneHotEncoder(handle_unknown='ignore')
enc.fit(X_train)
X_train = enc.transform(X_train)
X_test = enc.transform(X_test)

Question 2.3: Apply [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html) to the training data since there is class imbalance.

In [14]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)
X_test, y_test = sm.fit_resample(X_test, y_test)

Question 3.1: Train five machine learning [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html), [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html), [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), and [XGBClassifier](https://xgboost.readthedocs.io/en/latest/python/python_api.html#xgboost.XGBClassifier) based on the training data, and evaluate their performance on the test dataset. Use default hyperparameter values.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost.sklearn import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [17]:
#Build models (You can either do it combined or separate)

lr, sv, kn, dt, xg = LogisticRegression(random_state=42), SVC(random_state=42), KNeighborsClassifier(), DecisionTreeClassifier(random_state=42),XGBClassifier(random_state=42)
lr.fit(X_train, y_train), sv.fit(X_train, y_train), kn.fit(X_train, y_train), dt.fit(X_train, y_train), xg.fit(X_train, y_train);

In [18]:
for model in [lr, sv, kn, dt, xg]:
    print("\n",model.replace(")
    y_pred = model.predict(X_test)
    print("confusion matrix:\n",confusion_matrix(y_test, y_pred))
    #print("score:",model.score(X_test, y_pred))
    print("classification report:\n",classification_report(y_test,y_pred,target_names=["Machine Running","Machine Failed"]))
    print()


 LogisticRegression(random_state=42)
confusion matrix:
 [[3076  123]
 [2537  662]]
classification report:
                  precision    recall  f1-score   support

Machine Running       0.55      0.96      0.70      3199
 Machine Failed       0.84      0.21      0.33      3199

       accuracy                           0.58      6398
      macro avg       0.70      0.58      0.52      6398
   weighted avg       0.70      0.58      0.52      6398



 SVC(random_state=42)
confusion matrix:
 [[3191    8]
 [1402 1797]]
classification report:
                  precision    recall  f1-score   support

Machine Running       0.69      1.00      0.82      3199
 Machine Failed       1.00      0.56      0.72      3199

       accuracy                           0.78      6398
      macro avg       0.85      0.78      0.77      6398
   weighted avg       0.85      0.78      0.77      6398



 KNeighborsClassifier()
confusion matrix:
 [[1659 1540]
 [ 330 2869]]
classification report:
             

Questions 3.2:  Perform recursive feature elimination (3 features) on the dataset using a logistic regression classifier with max_iter= 1000, random_state=5.  Any difference in the results? Explain.

In [23]:
from sklearn.feature_selection import RFE
rfe = RFE(estimator=LogisticRegression(max_iter= 1000, random_state=5), n_features_to_select=3)

In [24]:
rfe.fit(X_train,y_train)
X_train, X_test = rfe.predict(X_train).reshape(-1, 1), rfe.predict(X_test).reshape(-1, 1)

In [25]:
lr, sv, kn, dt, xg = LogisticRegression(random_state=42), SVC(random_state=42), KNeighborsClassifier(), DecisionTreeClassifier(random_state=42),XGBClassifier(random_state=42)
lr.fit(X_train, y_train), sv.fit(X_train, y_train), kn.fit(X_train, y_train), dt.fit(X_train, y_train), xg.fit(X_train, y_train);

In [28]:
for model in [lr, sv, kn, dt, xg]:
    print(model)
    y_pred = model.predict(X_test)
    print("confusion matrix:\n",confusion_matrix(y_test, y_pred))
    #print("score:",model.score(X_test, y_pred))
    print("classification report:\n",classification_report(y_test,y_pred,target_names=["Machine Running","Machine Failed"]))

LogisticRegression(random_state=42)
confusion matrix:
 [[3196    3]
 [3113   86]]
classification report:
                  precision    recall  f1-score   support

Machine Running       0.51      1.00      0.67      3199
 Machine Failed       0.97      0.03      0.05      3199

       accuracy                           0.51      6398
      macro avg       0.74      0.51      0.36      6398
   weighted avg       0.74      0.51      0.36      6398

SVC(random_state=42)
confusion matrix:
 [[3196    3]
 [3113   86]]
classification report:
                  precision    recall  f1-score   support

Machine Running       0.51      1.00      0.67      3199
 Machine Failed       0.97      0.03      0.05      3199

       accuracy                           0.51      6398
      macro avg       0.74      0.51      0.36      6398
   weighted avg       0.74      0.51      0.36      6398

KNeighborsClassifier()
confusion matrix:
 [[3196    3]
 [3113   86]]
classification report:
                  pre

##### What's interesting is there's marginal improvements across the board but also some drawbacks in terms of precision. So basically the changes in improvement appear to offset such that it's not really worth this extra effort as far as precision goes. There's the case of the f1-scores on the otherhand which are incredibly different, but only for when the machine fails, the same goes for the recal values. The deeper implications of this escape me at the moment but the eliminated features surely contribute to this as it provides less data for the model to process and ultimately leads to this lower f1-score which can be interpretted as the model giving less accuracate predictions.

Q.4. Create a new text cell in your Notebook: Complete a 50-100 word summary (or short description of your thinking in applying this week's learning to the solution) of your experience in this assignment. Include:
What was your incoming experience with this model, if any? what steps you took, what obstacles you encountered. how you link this exercise to real-world, machine learning problem-solving. (What steps were missing? What else do you need to learn?) This summary allows your instructor to know how you are doing and allot points for your effort in thinking and planning, and making connections to real-world work. 

##### I wasn't able to spend as much time on this assignment as previous assingments so I'm not incredibly proud of it, I didn't have any experience with the model at all so everything was new other than what I've learned in previous assignments. I used the sklearn documentation extensively for researching purposes which proved to be sufficient for cases where I just needed a reminder or even an in depth understanding of a model's attributes. I think this relates to a real world instance of applying ML in the sense that several models were considered and applied as well as some preprocessing steps all in effort to identify whether or not a machine has failed. This could be incredibly useful in an enviroment where a large array of machines are being used but are dated and it may be costly to install sensors to know exactly which machine breaks or how many there are, this would allow one to estimate the number of broken machines which is a potential temporary substitute for upgrading the hardware on the machines so this information could be automatically reported (which again may be costly or even impossible...). I'm wondering if this would be useful in determining when and/or which specific hardware components in a satelite might break or fail after some t time, I wonder if it could be useful in designing them potentially (this is just a stab in the dark suggestion).